# v9_b Step 3c: Match v9_b Step 3 features and standards to public iPSC features

This notebook applies the same matching rules as v9_b Step 3b Thyroid to the STS and 07242024 public iPSC projects. Each study has Cells and Supernatant tables for DnCl and DnHz. It matches all v9_b Step 3 khipu candidates and current authentic standards at m/z <10 ppm and RT difference <15 s. DnHz target RT is shifted by −120 s in memory, while its original RT is retained. All m/z/RT matches are kept; no sample intensity, genotype, treatment, or MS2 filter is applied. Results are stored separately for each of the eight input tables.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from asari.tools.file_io import read_features_from_asari_table
from asari.tools import match_features as mf

ROOT = Path.cwd().resolve()
assert (ROOT / 'results/step3_khipus').is_dir(), 'Run this notebook from the v9_b directory'
INPUT_DIR = ROOT / 'input/step3c_public_ipsc'
OUTPUT_DIR = ROOT / 'results/step3c_public_ipsc_feature_matching'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def write_matched_features_to_json(matched, dict_query, dict_features, output_file):
    matched_features = []
    for feature_id, matched_list in matched.items():
        matched_feature = dict_query.get(feature_id)
        if matched_feature:
            matched_features.append({
                'Feature': matched_feature,
                'Matched_libs': [dict_features[lib_id] for lib_id in matched_list if lib_id in dict_features]
            })
    with open(output_file, 'w') as f:
        json.dump(matched_features, f, indent=4)
    return matched_features

## Load v9_b Step 3 khipu features

In [ ]:
candidates_dir = ROOT / 'results/step3_khipus'
DnCl_candidates = json.loads((candidates_dir / 'DnCl_pellets_features.json').read_text()) + json.loads(
    (candidates_dir / 'DnCl_supernatant_features.json').read_text())
DnHz_candidates = json.loads((candidates_dir / 'DnHz_pellets_features.json').read_text()) + json.loads(
    (candidates_dir / 'DnHz_supernatant_features.json').read_text())

# Keep both sources when they contain the same feature ID.
from collections import Counter
for candidates in (DnCl_candidates, DnHz_candidates):
    duplicate_ids = {feature_id for feature_id, count in Counter(f['id'] for f in candidates).items() if count > 1}
    for feature in candidates:
        if feature['id'] in duplicate_ids:
            feature['original_id'] = feature['id']
            feature['id'] = f"{feature['source']}:{feature['id']}"
    assert len(candidates) == len({f['id'] for f in candidates})

print(f'Read {len(DnCl_candidates)} DnCl and {len(DnHz_candidates)} DnHz khipu candidates')

## Expand authentic-standard RT entries

In [ ]:
manual_annotation_path = ROOT / 'input/step3c_public_ipsc/Authentic_standard_list.csv'
manu_DnCl, manu_DnHz = [], []
for _, row in pd.read_csv(manual_annotation_path).iterrows():
    feature_id = row['No']
    mz = row['Exact_mass_[M+H]+_after_derivatization']
    rtList = row['RT_min'].split(';')
    lib_id = row['KEGG_number']
    method = row['Derivatization']
    if 'DnCl' in method:
        for rt_index, rt in enumerate(rtList, start=1):
            rt_min = float(rt)
            manu_DnCl.append({'id': f'STD:{feature_id}:RT{rt_index}', 'standard_id': feature_id,
                              'standard_name': row['Name'], 'standard_rt_min': rt_min,
                              'mz': float(mz), 'rtime': rt_min * 60, 'lib_id': lib_id})
    elif 'DnHz' in method:
        for rt_index, rt in enumerate(rtList, start=1):
            rt_min = float(rt)
            manu_DnHz.append({'id': f'STD:{feature_id}:RT{rt_index}', 'standard_id': feature_id,
                              'standard_name': row['Name'], 'standard_rt_min': rt_min,
                              'mz': float(mz), 'rtime': rt_min * 60, 'lib_id': lib_id})

queries = {'DnCl': DnCl_candidates + manu_DnCl, 'DnHz': DnHz_candidates + manu_DnHz}
query_by_id = {deriv: {f['id']: f for f in items} for deriv, items in queries.items()}
for deriv, items in queries.items():
    assert len(query_by_id[deriv]) == len(items)
    print(deriv, len(items), 'queries including', len(manu_DnCl if deriv == 'DnCl' else manu_DnHz), 'standard RT entries')

## Match each iPSC feature table in both directions

In [ ]:
studies = ['STS', '07242024_CYP19A1_SLC22A11_CYP11A1_HSD17B1_ABCG2']
matrices = ['Cells', 'Supernatant']
derivations = ['DnCl', 'DnHz']
summary = []
for study in studies:
    for matrix in matrices:
        for deriv in derivations:
            table_path = INPUT_DIR / study / matrix / f'{deriv}_full_Feature_table.tsv'
            num_samples, target_features = read_features_from_asari_table(table_path.read_text())
            target_by_id = {f['id']: f for f in target_features}
            assert len(target_by_id) == len(target_features), table_path

            # Align DnHz target RT to the candidate tables, retaining the raw value.
            if deriv == 'DnHz':
                for feature in target_features:
                    feature['rtime_raw_s'] = feature['rtime']
                    feature['rtime'] -= 120

            result_dir = OUTPUT_DIR / study / matrix / deriv
            result_dir.mkdir(parents=True, exist_ok=True)
            forward = mf.list_match_lcms_features(queries[deriv], target_features,
                                                   mz_ppm=10, rt_tolerance=15)
            forward_rows = write_matched_features_to_json(
                forward, query_by_id[deriv], target_by_id,
                result_dir / 'v9_candidates_matched_ipsc.json')

            reverse = mf.list_match_lcms_features(target_features, queries[deriv],
                                                   mz_ppm=10, rt_tolerance=15)
            reverse_rows = write_matched_features_to_json(
                reverse, target_by_id, query_by_id[deriv],
                result_dir / 'ipsc_matched_v9_candidates.json')

            forward_pairs = sum(len(row['Matched_libs']) for row in forward_rows)
            reverse_pairs = sum(len(row['Matched_libs']) for row in reverse_rows)
            summary.append({
                'study': study, 'matrix': matrix, 'derivatization': deriv,
                'input_feature_count': len(target_features), 'sample_column_count': num_samples,
                'query_count': len(queries[deriv]),
                'matched_query_count': len(forward_rows),
                'matched_ipsc_feature_count': len(reverse_rows),
                'forward_matched_pair_count': forward_pairs,
                'reverse_matched_pair_count': reverse_pairs,
            })
            print(study, matrix, deriv, 'input=', len(target_features),
                  'queries_matched=', len(forward_rows), 'ipsc_features_matched=', len(reverse_rows),
                  'pairs=', forward_pairs, '/', reverse_pairs)

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUTPUT_DIR / 'summary_by_table.tsv', sep='\t', index=False)
summary_df